In [177]:
%reset -f

In [178]:
from get_data import *
from online_forecasting import *
from dash_plotter import DashRealTimePlotter
import warnings
import logging

import pandas as pd
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
import pickle
import os


import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report

import re
from sklearn.preprocessing import RobustScaler
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras import callbacks
import pickle
import numpy as np

warnings_logger = logging.getLogger('warnings')
warnings_logger.setLevel(logging.WARNING)
warning_handler = logging.FileHandler('warnings.log')
warning_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
warnings_logger.addHandler(warning_handler)

def warning_handler_func(message, category, filename, lineno, file=None, line=None):
    warnings_logger.warning(f"{category.__name__}: {message} (File: {filename}, Line: {lineno})")

warnings.showwarning = warning_handler_func

In [179]:
# 0. Get the same Data as in Forecasting part
df_removed_nans_forecasting, df_removed_nans_classification = get_data()

Getting values...
Numeric columns: (39435, 48)
Status columns: (39435, 22)
Preprocessing data...


In [180]:
# 0. Merge data
df = pd.merge(df_removed_nans_forecasting, df_removed_nans_classification, on=['timestamp'])

In [181]:
# 1. Encode Column Names
# def extract_int(string_input):
#     for letter in string_input:
#         if letter.isdigit():
#             return int(letter)
#     return None

def extract_port_numbers(string_input):
    """Extract all numbers from interface string, handling multi-digit ports"""
    numbers = re.findall(r'\d+', string_input)
    return ''.join(numbers) if numbers else None

encoded_columns = {}
unique_ports = []

for column in df.columns:
    if "interface" in column.lower() and "/" in column:
        port_strings = column.split("/")
        port_numbers = []
        for port_string in port_strings:
            #num = extract_int(port_string)
            num = extract_port_numbers(port_string)
            if num is not None:
                port_numbers.append(str(num))
        encoded_columns[column] = "".join(port_numbers)
    elif ": operational status" in column.lower() and "/" in column:
        encoded_columns[column] = column

for idx, name in encoded_columns.items():
    if ": operational status" in idx.lower():
        map_name = f"Status {name}"
        encoded_columns[idx] = map_name
    elif "bits sent" in idx.lower():
        map_name = f"Bits Sent {name}"
        encoded_columns[idx] = map_name
    elif "bits received" in idx.lower():
        map_name = f"Bits Received {name}"
        encoded_columns[idx] = map_name

    if int(name) not in unique_ports:
        unique_ports.append(int(name))

# Update DataFrame with encoded column names
df.rename(columns=encoded_columns, inplace=True)

In [182]:
# map ports to start from 1
mapped_ports = {}
i = 1
for port in unique_ports:
    mapped_ports[port] = i
    i+=1

In [183]:
# 2. Encode Labels
temperature_threshold = 50
cpu_threshold = 30

temp_bit = 8
cpu_bit = 10

def encode_labels(df, down_value = 2, temperature_threshold = temperature_threshold, cpu_threshold = cpu_threshold, temp_bit = temp_bit, cpu_bit = cpu_bit):
    status_columns_classification = [col for col in df.columns if "Status" in col and "interface" not in col]
    temperature_columns = [col for col in df.columns if "temperature" in col.lower()]
    cpu_columns = [col for col in df.columns if "cpu" in col.lower()]

    # pandas method that applies function to each row in DataFrame, one at a time
    def create_binary_encoding(row):
        down_ports = []

        for col in status_columns_classification:
            port_num = int(col.replace('Status ',''))
            mapped_port = mapped_ports[port_num]
            
            status_value = row[col]
            if status_value == down_value:
                down_ports.append(mapped_port)
        
        # encode statuses 
        binary_value = 0
        for port in down_ports:

            # Initial: binary_value = 0 (00000000)

            # Port 1 down:
            # binary_value |= (1 << 0)  →  0 | 1  →  00000001 (decimal 1)

            # Port 3 down:
            # binary_value |= (1 << 2)  →  1 | 4  →  00000101 (decimal 5)

            # Port 5 down:
            # binary_value |= (1 << 4)  →  5 | 16 →  00010101 (decimal 21)
            
            binary_value |= (1 << port - 1)

        # simple temperature check (OK/NOK)
        # if either of the temperature sensors is above threshold, set bit
        for temperature_column in temperature_columns:
            if row[temperature_column] > temperature_threshold:
                binary_value |= (1 << temp_bit)

        for cpu_column in cpu_columns:
            if row[cpu_column] > cpu_threshold:
                binary_value |= (1 << cpu_bit)

        return binary_value

    df_labeled = df.copy()
    df_labeled['Label'] = df.apply(create_binary_encoding, axis=1)
    return df_labeled

df_labeled = encode_labels(df)

In [184]:
# 3. Label Decoder
def decode_label(label, mapped_ports, temp_bit=temp_bit, cpu_bit=cpu_bit):
    """
    Decodes the integer label into a dictionary indicating which ports are down,
    and whether temperature/cpu alarms are set.
    """
    result = {
        "down_ports": [],
        "temperature_alarm": False,
        "cpu_alarm": False
    }
    
    # Create reverse mapping from mapped port numbers back to original port numbers
    reverse_mapped_ports = {v: k for k, v in mapped_ports.items()}
    
    # Check each possible mapped port position
    for mapped_port in range(1, len(mapped_ports) + 1):
        if label & (1 << (mapped_port - 1)):
            original_port = reverse_mapped_ports[mapped_port]
            result["down_ports"].append(original_port)
    
    # Check temperature bit
    if label & (1 << temp_bit):
        result["temperature_alarm"] = True
    
    # Check cpu bit
    if label & (1 << cpu_bit):
        result["cpu_alarm"] = True
    
    return result

label_to_name = {}
labels = df_labeled['Label'].unique()

for label in labels:
    label_to_name[label] = decode_label(label, mapped_ports, temp_bit=temp_bit, cpu_bit=cpu_bit)

In [185]:
# 4. Balance Classes by Downsampling Majority Classes
def merge_small_classes(df, label_col='label', threshold=10, other_label='other'):
    class_counts = df[label_col].value_counts()
    small_classes = class_counts[class_counts < threshold].index
    df_merged = df.copy()
    df_merged[label_col] = df_merged[label_col].apply(lambda x: other_label if x in small_classes else x)
    return df_merged

merged_labeled_df = merge_small_classes(df_labeled, label_col='Label', threshold=3000, other_label='0000')
print(merged_labeled_df['Label'].value_counts())
# unify datatype for label column
merged_labeled_df['Label'] = merged_labeled_df['Label'].astype(int)

Label
21626    10583
24187    10315
0000      6349
29818     4188
Name: count, dtype: int64


In [186]:
# 5. Merge Small Classes
def balance_classes(df, label_col='label', random_state=42):
    class_counts = df[label_col].value_counts()
    min_count = class_counts.min()

    balanced_frames = []
    for cls in class_counts.index:
        cls_df = df[df[label_col] == cls]
        balanced_cls_df = resample(cls_df, 
                                   replace=False, 
                                   n_samples=min_count, 
                                   random_state=random_state)
        balanced_frames.append(balanced_cls_df)
    
    # Concatenate and shuffle
    balanced_df = pd.concat(balanced_frames).sample(frac=1, random_state=random_state).reset_index(drop=True)
    return balanced_df

df_balanced_labeled = balance_classes(df = merged_labeled_df, label_col = 'Label')
print(df_balanced_labeled['Label'].value_counts())

Label
21626    4188
0        4188
24187    4188
29818    4188
Name: count, dtype: int64


In [187]:
# 6. Prepare Data for Training

# simple approach of labeling for classification model
df_classification_input = df_balanced_labeled.copy()
unique_labels = sorted(df_classification_input['Label'].unique())
label_to_index = {label: index for index, label in enumerate(unique_labels)}
df_classification_input['Label'] = df_classification_input['Label'].map(label_to_index)

features_classification = df_classification_input[df_classification_input.columns[1:-1]].values
labels_classification = df_classification_input[df_classification_input.columns[-1]].values

# RobustScaler is better for network data (handles outliers)
scaler = RobustScaler()
features_scaled = scaler.fit_transform(features_classification)

# save the scaler for later use

with open('classification_model/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

def create_windows(X, y, window_size):
    Xs, ys = [], []
    for i in range(len(X) - window_size + 1):
        Xs.append(X[i:i+window_size])
        ys.append(y[i+window_size-1])  # label from last time step in window
    return np.array(Xs), np.array(ys)

window_size = 6
X_seq, y_seq = create_windows(features_scaled, labels_classification, window_size)

X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_seq,
    test_size=0.2,
    stratify=y_seq,   # ensures same label distribution in both
    random_state=42
)

In [188]:
# 7. Build and Train Model

num_classes = len(df_classification_input['Label'].unique())
model = models.Sequential([
    layers.Conv1D(64, kernel_size=3, activation='relu', input_shape=(window_size, X_seq.shape[2])),
    layers.BatchNormalization(),
    layers.Conv1D(128, kernel_size=3, activation='relu'),
    layers.BatchNormalization(),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# callbacks for better training
callbacks_list = [
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        'classification_model/best_model.h5',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

# updated training call
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=10,  # More epochs with early stopping
    batch_size=16,
    callbacks=callbacks_list,
    verbose=1
)

y_pred = model.predict(X_test).argmax(axis=1)

test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

Epoch 1/10
670/670 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3572 - loss: 1.3802
Epoch 1: val_loss improved from None to 1.60232, saving model to classification_model/best_model.h5
670/670 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.4065 - loss: 1.2432 - val_accuracy: 0.2582 - val_loss: 1.6023 - learning_rate: 0.0010
Epoch 2/10
666/670 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5552 - loss: 0.9664
Epoch 2: val_loss improved from 1.60232 to 1.59940, saving model to classification_model/best_model.h5
670/670 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.6011 - loss: 0.8796 - val_accuracy: 0.3910 - val_loss: 1.5994 - learning_rate: 0.0010
Epoch 3/10
668/670 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6752 - loss: 0.7247
Epoch 3: val_loss improved from 1.59940 to 1.50197, saving model to classification_model/best_model.h5
670/670 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.6884 - loss: 0.6947 - val_accuracy: 0.4101 - val_loss: 1.5020 - learning_rate: 0.0010
Epoch 4/10


In [191]:
# 8. Save Model and Encoders
model.save('classification_model/final_model.h5')
encoder_data = {}
encoder_data['label_to_index'] = label_to_index
encoder_data['label_to_name'] = label_to_name
with open('classification_model/encoders.pkl', 'wb') as f:
    pickle.dump(encoder_data, f)
